# MODULE 46 — FACIAL RECOGNITION

# MobileFaceNet Architecture

🔥 **Must Understand — Conceptual, Architectural and Mathematical**

Let's properly understand MobileFaceNet from the input image all the way to the final facial embedding.

We will study it at three levels:

1. **Conceptual:** Why does each component exist?
2. **Architectural:** How does data flow through every layer?
3. **Mathematical:** How do convolutions, bottlenecks and GDConv actually work?

We'll use the original MobileFaceNet architecture from the 2018 research paper:

**MobileFaceNets: Efficient CNNs for Accurate Real-Time Face Verification on Mobile Devices**

Authors: Sheng Chen, Yang Liu, Xiang Gao and Zhen Han.

[Original research paper — arXiv](https://arxiv.org/abs/1804.07573)

---

## 1. What is MobileFaceNet trying to achieve?

First, let's understand the problem.

Suppose we give our network two images:

**Face A**

![Face A](https://images.openai.com/static-rsc-4/AKs3Za7fV5CgYYnAU5jWzyqbr5iF352qqKb39s7UlBhGmmnZBPa-64C37BTigBns0knNVqq73pB0y2GtM_a5vAGmskKUqDF3mTyXQMA5-clAuOSRxWlQMCeQm5Vb1GyRxhUr5fnowISHQGr7-ExS9nrG3_m_uk9eaJSVGNMtw1w?purpose=inline)

**Face B**

![Face B](https://images.openai.com/static-rsc-4/IGrZJ-pdItZQBU9cTetPcpHjXXxD5F0U8XpRK9sVk1GG6o5bXUaW8Wmh84J-twDiAxY3gYsk8Gz2gGM0AcV1cJ7_soe2z_9bDB-9bE-7JMrisYzVEZCWGgZBbqNnxkWw7mhyw3_dBPWo3ksJAFS4f_g9Anf9cc6J6SUiJ-8295s?purpose=inline)

The images might have:

- Different lighting.
- Different facial expressions.
- Different camera angles.
- Different backgrounds.
- Different image quality.

But if they belong to the same person, our model should generate similar facial representations.

The objective of MobileFaceNet is:

**Convert a face image into a compact numerical representation that preserves identity-related information.**

This numerical representation is called a **facial embedding**.

For example:

$$
I \longrightarrow f(I) \longrightarrow [0.12,-0.34,0.81,\ldots]
$$

Here, \(f(I)\) represents the MobileFaceNet feature extraction function.

The original design produces a **128-dimensional embedding**.

---

## 2. Complete MobileFaceNet architecture

![MobileFaceNet architecture diagram](https://images.openai.com/static-rsc-4/_OxnkMzlm1qcZ062dLAH0pqb9VLZG5lU-ElBeAaU5UwLATSYvBkGMzHdfSogWKkbI63hqbrE5xLXURLIhH3gOvK4GpD7js8PAY_XCTaHMxldsdFU7utF4mpw0Sdk9J6F7N_rQxd-Zd5uGuzsHVboyLLr7yPDeU9k2YVMA-NC9wI?purpose=inline)

[Original diagram source — CSDN](https://blog.csdn.net/weixin_39875161/article/details/91535040)

Let's understand the complete flow.

### MobileFaceNet — Architecture flow

```text
Input image
112 × 112 × 3
       |
       v
Initial Conv 3 × 3
56 × 56 × 64
       |
       v
Depthwise Conv 3 × 3
56 × 56 × 64
       |
       v
Bottleneck stages
       |
       v
7 × 7 × 128
       |
       v
1 × 1 Conv
7 × 7 × 512
       |
       v
GDConv 7 × 7
1 × 1 × 512
       |
       v
Final Linear Conv 1 × 1
       |
       v
128-D
       |
       v
Facial embedding
```

Now let's understand every component.

---

## 3. Understanding the input layer

Our input image:

$$
X \in \mathbb{R}^{112\times112\times3}
$$

What do these dimensions represent?

| Dimension | Meaning |
|---|---|
| 112 | Image height |
| 112 | Image width |
| 3 | RGB channels |

The model expects a face image that has already been detected, aligned and resized.

### Why 112 × 112?

The network was designed around this relatively small face resolution to reduce computation while retaining useful facial information.

It doesn't mean that the original photograph must be 112 × 112. Face detection and alignment happen before this network.

---

## 4. Initial convolution — Feature extraction

![Convolutional Neural Network illustration](https://images.openai.com/static-rsc-4/qaHBlerD5WPETZCcH9TTsNu4pLjLPCj-VARMCXUZtXc2pZ7GFl8gAtEBQnQ6nVUl-sk3CeWR_XhZeEyllazodYU4OgMmejC3YofsMsBZnwnkgSmD_sr4vZ5SV8nLSLW8Sbs6aI2HWhP5-9XbI5L28uXBK9RYYAKhRUILXYaCmno?purpose=inline)

[Source — Machine Learning Theory](https://machinelearningtheory.org/docs/Deep-Learning/cnn/)

The first layer is:

$$
Conv(3\times3,\;64,\;stride=2)
$$

**Input:**

$$
112\times112\times3
$$

**Output:**

$$
56\times56\times64
$$

### What happens here?

The convolutional layer applies 64 different filters to the input image.

Each filter learns to recognize different visual patterns.

For example:

- Horizontal edges.
- Vertical edges.
- Diagonal patterns.
- Basic textures.
- Local intensity changes.

The stride is 2, meaning the filter moves two pixels at a time.

This reduces the spatial dimensions from 112 × 112 to 56 × 56.

### Why reduce the image size?

Smaller feature maps mean fewer subsequent computations.

We are progressively compressing spatial information while extracting useful features.

---

## 5. Depthwise convolution

![MobileNet depthwise convolution illustration](https://images.openai.com/static-rsc-4/Prx8svRxZoBP9VtAVFcW7ZgA4ih-bPf9RyoxOKkf3IsfnocldUXc_wPDL-3r3D8CwsDvT068KD06iCmHno1E4qyFXUYdrN03XsKdGpewu-7oa4_Mn1tSBdmdgVZHT5RzBnDdJ2veHEVX3i8nfyLZYnEOuuB1qTOnfbEL3wKkyGw?purpose=inline)

[Source — Gaussian37](https://gaussian37.github.io/dl-concept-mobilenet/)

The next operation is:

$$
DepthwiseConv(3\times3,\;stride=1)
$$

**Input:**

$$
56\times56\times64
$$

**Output:**

$$
56\times56\times64
$$

This is one of the most important concepts in lightweight CNNs.

### 5.1 Traditional convolution

Suppose we have 64 input channels.

A traditional convolution filter operates across all input channels and combines them to produce an output channel.

For 64 input channels and 64 output channels, the number of kernel weights is:

$$
3\times3\times64\times64=36,864
$$

### 5.2 Depthwise convolution

Instead, depthwise convolution applies one spatial filter independently to each input channel.

For 64 channels:

$$
3\times3\times64=576
$$

**64× fewer kernel weights** compared with the corresponding standard convolution.

This is the basic idea behind depthwise convolution.

But there is a limitation: depthwise convolution alone doesn't mix information between channels.

That's why we need pointwise convolution.

We'll see that inside the bottleneck block.

---

## 6. The Bottleneck block — The heart of MobileFaceNet

<badge color="success">Most important component</badge>

This is where MobileFaceNet borrows a major idea from MobileNetV2.

MobileFaceNet uses **Inverted Residual Bottleneck blocks**.

Let's understand why they're called that.

### 6.1 Traditional residual block vs inverted residual block

In a traditional residual block, the network often:

1. Expands the number of feature channels.
2. Processes features.
3. Reduces the number of channels.

In MobileNetV2-style inverted residual blocks, the process is:

1. Expand channels.
2. Apply depthwise convolution.
3. Project channels back down.

![Inverted residual block illustration](https://images.openai.com/static-rsc-4/VldT83Igkzrcqth9LHVO4BW4tQODryBg5ACxkbAtbYY9borLZN5dv5KQCGC7yqrjI3SADL6KnHsGuu40Ulr4l7OfXRuvBjv8RauXtNpa-G2oVMFMkV7pPMEIm1T2lIUaU3PNPiDtup9e-3_JUa3RkNg7cBu-INnDNzzBeCrGVWg?purpose=inline)

[Source — Scientific Reports](https://www.nature.com/articles/s41598-026-37743-0)

The basic structure is:

$$
Input \rightarrow Expand \rightarrow Depthwise \rightarrow Project \rightarrow Output
$$

Let's understand the three main operations.

### 6.2 Expansion — Pointwise convolution

Suppose the input is:

$$
28\times28\times64
$$

The expansion factor is:

$$
t=2
$$

The network increases the channels:

$$
64\times2=128
$$

**Output:**

$$
28\times28\times128
$$

How?

Using a 1 × 1 pointwise convolution.

### Why do we expand channels?

Because each channel contains different feature information. Expanding the channel dimension gives the network more intermediate feature representations to work with.

Think of it as increasing the number of feature-processing pathways.

### 6.3 Depthwise convolution — Spatial feature extraction

Now we apply:

$$
DepthwiseConv(3\times3)
$$

**Input:**

$$
28\times28\times128
$$

**Output:**

$$
28\times28\times128
$$

This operation extracts spatial information independently from each channel.

For example, different channels might respond to:

- Eye-region patterns.
- Nose structures.
- Mouth contours.
- Facial edges.
- Texture variations.

These are illustrative interpretations; the network learns its own feature representations rather than assigning a fixed facial part to every channel.

### 6.4 Projection — Linear pointwise convolution

Now we reduce the channels again.

**Input:**

$$
28\times28\times128
$$

Projection:

$$
1\times1\ Conv
$$

**Output:**

$$
28\times28\times64
$$

### Why reduce channels?

To keep the network computationally efficient.

The resulting feature representation is compact again.

### Why is the projection linear?

The projection layer doesn't use a nonlinear activation after its convolution.

It is commonly called a **linear bottleneck**.

The motivation is to avoid destroying useful information when projecting into a lower-dimensional feature space.

### 6.5 Residual connection

![MobileNetV2 residual connection example](https://images.openai.com/static-rsc-4/50MJRK3qZkRuQ0CrxIlHVmg6XPLbeDlhjDejkZoKBdN6yqf6-bGuYl8reBVbfauvPpl9LOBTpVw8FAFwPHhy1wFwkyCE4NZrO1lKdG_EVvDberOStevX16bvJjCb4359jX4kb9Y5pGZjADZX_3gHzA7PK96zAYO-xVhx-_pF38s?purpose=inline)

[Source — GitHub MobileNetV2](https://github.com/TenTen-Teng/MobileNet-V2)

Now something interesting happens.

If the input and output have identical shapes, we can add them:

$$
Y=F(X)+X
$$

Here:

- \(X\) = original input.
- \(F(X)\) = transformed feature representation.
- \(Y\) = final output.

### Why do we do this?

Benefits:

1. Helps gradient propagation.
2. Reduces degradation in deeper networks.
3. Preserves information from earlier layers.
4. Helps the network learn residual transformations.

One important condition: the shortcut addition requires compatible spatial dimensions and channel counts. When a block downsamples or changes channels, the ordinary identity shortcut cannot be used directly.

### 6.6 What is PReLU?

MobileFaceNet uses PReLU as its activation function.

![PReLU activation function illustration](https://images.openai.com/static-rsc-4/2K3UX_NrRNnJM2MxwPuZqk7muucmM5-klPszT1LS3mqCRqAFm_GGATcFtu2Fjh0Nr5XvrlOoRaKXwxE4A7kDOw0aCps2qdDRMJfAGk-nwWi0eU-bDGrUb75muyXGwVNGrefOa9ddmX3lK6OqLc6-5hUce0dVli0EINNvQurFQDQ?purpose=inline)

[Source — Paper Comprehension](https://yuk068.github.io/2026/05/08/motion-planning-network-paper-review)

ReLU is:

$$
f(x)=\max(0,x)
$$

It converts negative values to zero.

PReLU is:

$$
f(x)=
\begin{cases}
x & x>0\\
ax & x\leq0
\end{cases}
$$

Here, \(a\) is a learnable parameter.

Unlike ordinary ReLU, PReLU allows a learnable negative slope.

This can help preserve information carried by negative activations.

Also remember: Batch Normalization is used throughout the architecture during training to normalize intermediate activations and stabilize optimization.

---

## 7. Complete bottleneck stages

Now let's put the individual bottleneck blocks together.

This is the architectural progression from the original MobileFaceNet design.

### Complete architecture configuration

| Stage | Expansion (t) | Output channels (c) | Repetitions (n) | First stride (s) | Output shape |
|---|---:|---:|---:|---:|---|
| Input | — | 3 | — | — | 112 × 112 × 3 |
| Initial Conv | — | 64 | 1 | 2 | 56 × 56 × 64 |
| Depthwise Conv | — | 64 | 1 | 1 | 56 × 56 × 64 |
| Bottleneck 1 | 2 | 64 | 5 | 2 | 28 × 28 × 64 |
| Bottleneck 2 | 4 | 128 | 1 | 2 | 14 × 14 × 128 |
| Bottleneck 3 | 2 | 128 | 6 | 1 | 14 × 14 × 128 |
| Bottleneck 4 | 4 | 128 | 1 | 2 | 7 × 7 × 128 |
| Bottleneck 5 | 2 | 128 | 2 | 1 | 7 × 7 × 128 |

Important terminology:

- `t` = expansion factor.
- `c` = output channels.
- `n` = number of repeated blocks.
- `s` = stride of the first block in that stage.

The remaining blocks in a repeated stage generally use stride 1.

### Why does the network progressively reduce spatial dimensions?

Look at the progression:

```text
56 × 56
64 channels
     |
     v
28 × 28
64 channels
     |
     v
14 × 14
128 channels
     |
     v
7 × 7
128 channels
```

Illustration of spatial downsampling, not a measure of total computational work.

As the spatial dimensions decrease, the network can process increasingly abstract feature representations at a lower computational cost.

At the same time, it maintains or increases channel capacity to represent more complex patterns.

---

## 8. Final convolution — 512 channels

After the bottleneck stages:

$$
X\in\mathbb{R}^{7\times7\times128}
$$

We apply:

$$
Conv(1\times1,\;512)
$$

**Output:**

$$
7\times7\times512
$$

### Why 512 channels?

This creates a richer feature representation before the final spatial aggregation.

Each of the 512 channels contains a learned feature map.

Now comes one of MobileFaceNet's most interesting innovations.

---

## 9. Global Depthwise Convolution (GDConv)

<badge color="success">Important architectural innovation</badge>

![Global depthwise convolution illustration](https://images.openai.com/static-rsc-4/eJ0wQpUfnHi0ZXRpQM5OD--xlQh3c_vEn1tFuz0pVPWn33XZjXME6fBq3MT9B48Fx6_25zcrRCF6uWlHOvdSkg2MtCX55M8DjmthC-tzQBfQxnW8RGQvgayjaF4TdDZArJhZjTOxl9laY0tIFV3Nz72OGWpWjKm1NuJBSvq92bg?purpose=inline)

[Source — MDPI](https://www.mdpi.com/2079-9292/10/21/2679)

Most traditional CNNs use Global Average Pooling (GAP) to convert spatial feature maps into a vector.

MobileFaceNet replaces this with **Global Depthwise Convolution**.

Let's understand the difference.

### 9.1 Global Average Pooling

Suppose one feature map is:

$$
F\in\mathbb{R}^{7\times7}
$$

Global Average Pooling calculates:

$$
G=\frac{1}{49}\sum_{i=1}^{7}\sum_{j=1}^{7}F_{i,j}
$$

It averages all 49 spatial values.

The same averaging operation is performed independently for every channel.

### The problem?

Every spatial location receives equal weight.

### 9.2 Global Depthwise Convolution

Instead of averaging, GDConv learns a separate spatial weighting kernel for each channel.

$$
G_m=\sum_{i=1}^{7}\sum_{j=1}^{7}K_{i,j,m}F_{i,j,m}
$$

Here:

- \(F\) = input feature map.
- \(K\) = learnable depthwise kernel.
- \(m\) = channel index.
- \(G_m\) = output value for that channel.

**Input:**

$$
7\times7\times512
$$

**Kernel:**

$$
7\times7\times512
$$

**Output:**

$$
1\times1\times512
$$

### Why is this useful?

Unlike average pooling, GDConv learns which spatial positions are more useful for each feature channel.

For example, a feature map might learn to assign different weights to different facial regions.

It doesn't explicitly know that a location is an eye or nose. It learns spatial importance from training.

This provides a learnable spatial aggregation mechanism without requiring a large fully connected layer.

The original research discusses this as an important design decision for compact face embeddings.

---

## 10. Final linear convolution — 128-dimensional embedding

We currently have:

$$
1\times1\times512
$$

The final operation is:

$$
Conv(1\times1,\;128)
$$

**Output:**

$$
1\times1\times128
$$

We now have our final facial embedding:

$$
E=[e_1,e_2,e_3,\ldots,e_{128}]
$$

### Example facial embedding

Illustrative values, not actual model output.

```text
 0.124
-0.352
 0.781
 0.093
-0.241
 0.614
 0.328
-0.119
 0.452
 0.086
-0.623
 0.193
 ...
 116 additional values
```

**Embedding dimension: 128**

### Why 128 dimensions?

Because the network is designed to compress the extracted facial information into a compact representation.

It is not storing a photograph or a list of facial measurements. It is learning a numerical feature space in which facial identities can be distinguished.

---

## 11. What happens during training?

Now we need to understand something very important.

MobileFaceNet itself extracts features. But how does it learn to recognize different people?

This is where our **ArcFace loss** comes in.

![ArcFace illustration](https://images.openai.com/static-rsc-4/tIDd-FQd6uMnobPWSC_S4z8cOJ7KbpIz-6HTt0hfiXRfFPTZtTiDEpqaTTUQeR3kaO4TQrRFmQJakSUJibZ-kErM50um6oo01kVpZl2pZAhoTmAHE0Yd1zE09kCTiD9JCHJIyr_xCKwT96kQWAwxEAZM3166e4Pn0l-76C2fo30?purpose=inline)

[ArcFace reference](https://wuterry.github.io/2018/09/12/ArcFace/)

Let's assume our training dataset contains 1,000 different identities.

For each training image:

1. Image enters MobileFaceNet.
2. MobileFaceNet generates a 128-dimensional embedding.
3. The embedding is passed to an ArcFace classification head.
4. The classifier calculates identity scores.
5. ArcFace loss compares predictions with the correct identity.
6. Backpropagation updates the trainable network parameters.

The ArcFace head uses the identity labels to encourage:

- Similar embeddings for the same identity.
- Better separation between different identities.
- Discriminative angular margins between identity classes.

Mathematically, ArcFace modifies the target-class angle:

$$
\cos(\theta_y+m)
$$

Where:

- \(\theta_y\) = angle for the correct identity.
- \(m\) = angular margin.

The objective encourages stronger separation between identities.

**Important distinction:** ArcFace is not a layer that replaces MobileFaceNet. It is a training objective used with a classification head.

During inference, we don't need the training classification head. We retain MobileFaceNet to generate embeddings.

---

## 12. What happens during actual face verification?

After training, we can compare two faces.

### Face verification pipeline

**Face A**

![Face A verification example](https://images.openai.com/static-rsc-4/Cuz6m9Fz8rfjdPTEN0dgN3toKJ8j3uHjAJZCUw-8vzxsELIhXF2ErrOwIqfjhzMuiFJwhYkFt8h6wQYS7b7gAaknuOZ55K9ASvJxIf742AIDUBH3kubVBgvt_CPOCd0QntOx6Fj2yI4nFoCfsjlS90deJq9nWZ-UAA5cWpfFtr4?purpose=inline)

**Face B**

![Face B verification example](https://images.openai.com/static-rsc-4/GR9FEzmKvaIY7zlNSINjppHUOvgfpq-RZXW6QNC9lTUF5XS_yGFyn_kf6p8Nb8-1kkYUKcrs_EA9f-IEN2FUMMywEsjSQynXLIUQM08NI5OCTM1T5D2KZI1120JLirb-q6_nsoGkG32KkdDCAszhmcQ5787MLVdc5wD2WefNo70?purpose=inline)

```text
          Face A                  Face B
             |                       |
             v                       v
        MobileFaceNet           MobileFaceNet
             |                       |
             v                       v
        Embedding A             Embedding B
             |                       |
             +-----------+-----------+
                         |
                         v
                 Cosine similarity
                         |
                         v
                Threshold comparison
                         |
                         v
              Same identity / Different
                     identity
```

Cosine similarity:

$$
S(A,B)=\frac{A\cdot B}{\|A\|\|B\|}
$$

The threshold must be selected using validation data, not arbitrarily chosen.

---

## 13. Complete architecture summary

Let's consolidate everything we learned.

| Layer | Operation | Output |
|---|---|---|
| Input | RGB image | 112 × 112 × 3 |
| 1 | Conv 3 × 3, stride 2 | 56 × 56 × 64 |
| 2 | Depthwise Conv 3 × 3 | 56 × 56 × 64 |
| 3 | Bottleneck (t=2, n=5) | 28 × 28 × 64 |
| 4 | Bottleneck (t=4, n=1) | 14 × 14 × 128 |
| 5 | Bottleneck (t=2, n=6) | 14 × 14 × 128 |
| 6 | Bottleneck (t=4, n=1) | 7 × 7 × 128 |
| 7 | Bottleneck (t=2, n=2) | 7 × 7 × 128 |
| 8 | Conv 1 × 1 | 7 × 7 × 512 |
| 9 | Linear GDConv 7 × 7 | 1 × 1 × 512 |
| 10 | Linear Conv 1 × 1 | 1 × 1 × 128 |

---

## 14. Important technical specifications

| Specification | Value |
|---|---|
| Input resolution | 112 × 112 |
| Input channels | 3 |
| Embedding dimension | 128 |
| Backbone parameters | Approximately 1 million |
| Model weight storage | Approximately 4 MB (FP32) |
| Main building block | Inverted residual bottleneck |
| Activation | PReLU |
| Normalization | Batch Normalization |
| Spatial aggregation | GDConv |
| Training objective | Can use ArcFace |
| Verification metric | Cosine similarity |

The original paper reports less than one million parameters for its compact MobileFaceNet model and reports **99.55% LFW verification accuracy** with ArcFace training.

---


# Task 2.4 — Understanding the MobileFaceNet Bottleneck Block

The bottleneck block is the most important building block in MobileFaceNet. It extracts facial features while keeping computation and memory requirements low.

## 1. Internal Structure

![MobileFaceNet bottleneck block reference](https://images.openai.com/static-rsc-4/WS-RRQvLu_lCSxdxBracWC3bu9Q37d1IVtosgm4ySsF-IOM5dfiUElRB5m5oAL8xGQNKEyoRb_dgtmW19RHtJB18mZ_xlHQYgN8kIaMocjgf70hhkAUU4XX8_S0x1pjebeoCIWTbTjSnBmGYEvd5tVZ70QosBOiT-YCOQa6GzoU?purpose=inline)

[Image source: Springer Nature](https://link.springer.com/article/10.1007/s11042-024-19521-0?utm_source=chatgpt.com)

A bottleneck block contains three main operations:

1. Expansion convolution (1×1)
2. Depthwise convolution (3×3)
3. Projection convolution (1×1)

Let's understand each operation using actual tensor dimensions.

## 2. Step-by-Step Tensor Processing

Suppose the input tensor is:

\\[
X = 64 \times 28 \times 28
\\]

### Step 1 — Expansion Convolution

Expansion factor:

\\[
t=2
\\]

A 1×1 convolution increases the channel count:

\\[
64 \times 28 \times 28
\rightarrow
128 \times 28 \times 28
\\]

Why? It creates a richer feature representation before spatial feature extraction.

### Step 2 — Depthwise Convolution

Apply a 3×3 depthwise convolution independently to each of the 128 channels.

\\[
128 \times 28 \times 28
\rightarrow
128 \times 28 \times 28
\\]

Important: Unlike standard convolution, depthwise convolution does not mix information between channels. Each channel gets its own spatial filter.

### Step 3 — Projection Convolution

Apply a 1×1 convolution to reduce the channels:

\\[
128 \times 28 \times 28
\rightarrow
64 \times 28 \times 28
\\]

This produces the output of the bottleneck block.

## 3. Residual Connection (Shortcut)

![MobileNetV2 inverted residual block reference](https://images.openai.com/static-rsc-4/PmG-Ovidg9rd5pnXfgwOMbtgbXtIKMe2gPcxM2yjIbQqLciXJc-kE5CKOsO2UnY6_gvOvJ8lV8toCJYaykPap-WEmRaVU5PnZnoqpr_IG-Jh-j6xhEDJm0n-4XgRe2zEPc7y9A5-C-b9UMNfQ8zp_1-IbawAh6PO0_XAL1bQ4-0?purpose=inline)

[Image source: Medium](https://medium.com/%40luis_gonzales/a-look-at-mobilenetv2-inverted-residuals-and-linear-bottlenecks-d49f85c12423?utm_source=chatgpt.com)

Since the input and output dimensions are identical, we can add the original input to the transformed output:

\\[
Y = F(X) + X
\\]

Where:

- \\(X\\) = original input
- \\(F(X)\\) = output after the three convolution operations
- \\(Y\\) = final output

Why use residual connections?

- Helps gradients flow through deeper networks.
- Makes optimization easier.
- Allows the network to preserve useful input features.

**Important:** A shortcut addition is possible only when the input and output dimensions match and the stride is 1. Downsampling blocks generally cannot use this direct identity shortcut.

## 4. What Happens When Stride = 2?

Consider an input:

\\[
64 \times 56 \times 56
\\]

The bottleneck performs:

| Operation | Output |
|---|---|
| Input | `64 × 56 × 56` |
| Expansion (1×1) | `128 × 56 × 56` |
| Depthwise (3×3, stride 2) | `128 × 28 × 28` |
| Projection (1×1) | `64 × 28 × 28` |

Notice that the spatial dimensions are reduced from 56×56 to 28×28.

Because the input and output spatial dimensions differ, the ordinary identity shortcut cannot be added directly.

---

# MobileFaceNet — Complete Forward-Pass Execution

## Input Image

**Input shape: 112 × 112 × 3**

Height × Width × Channels

![MobileFaceNet face input reference](https://images.openai.com/static-rsc-4/GgeKdnmLqxzo3VZKkgdOMZ46RuXLcuplvQrDDxDukmtkyt2-G4fphvVlZBdn46jodZM6Ra81l2BlI5n0uhZ2BoQ-oCUx3KmpJ5gtqbKVMs2ROTdWJ36IX6Scf9hXNEIxOyXxOk4TpYROkiJT0RZezS1eIUfUnG51PF_WvRorH68?purpose=inline)

[Image source: ielab](https://ielab.io/?utm_source=chatgpt.com)

One RGB face image, normalized before entering the network.

---

## 1. Complete Architecture Execution

The following uses the original MobileFaceNet configuration, with NHWC dimensions displayed for readability.

| Stage | Operation | Output shape |
|---|---|---|
| Input | RGB image | 112 × 112 × 3 |
| 1 | Conv 3×3, 64, stride 2 | 56 × 56 × 64 |
| 2 | Depthwise Conv 3×3, stride 1 | 56 × 56 × 64 |
| 3 | Bottleneck t=2, c=64, n=5, s=2 | 28 × 28 × 64 |
| 4 | Bottleneck t=4, c=128, n=1, s=2 | 14 × 14 × 128 |
| 5 | Bottleneck t=2, c=128, n=6, s=1 | 14 × 14 × 128 |
| 6 | Bottleneck t=4, c=128, n=1, s=2 | 7 × 7 × 128 |
| 7 | Bottleneck t=2, c=128, n=2, s=1 | 7 × 7 × 128 |
| 8 | Conv 1×1, 512 channels | 7 × 7 × 512 |
| 9 | Linear GDConv 7×7 | 1 × 1 × 512 |
| 10 | Linear Conv 1×1, 128 channels | 1 × 1 × 128 |

---

## 2. Detailed Processing at Every Stage

### Stage 1 — Initial Convolution

**Input**

112 × 112 × 3

**Operation**

- Conv2D
- Kernel: 3×3
- Filters: 64
- Stride: 2
- Padding: 1

**Output**

56 × 56 × 64

### Stage 2 — Initial Depthwise Convolution

**Input**

56 × 56 × 64

**Operation**

- Depthwise Conv2D
- Kernel: 3×3
- Channels: 64
- Stride: 1
- Padding: 1

**Output**

56 × 56 × 64

### Stage 3 — Bottleneck Group 1

**Repetitions: 5 blocks**

| Operation | Output shape |
|---|---|
| Input | 56 × 56 × 64 |
| Expansion 1×1 (t=2) | 56 × 56 × 128 |
| Depthwise 3×3 (first block stride 2) | 28 × 28 × 128 |
| Projection 1×1 | 28 × 28 × 64 |
| Residual addition (where applicable) | 28 × 28 × 64 |

The first block downsamples. The remaining four blocks use stride 1.

**Final output: 28 × 28 × 64**

### Stage 4 — Bottleneck Group 2

**Repetitions: 1 block**

| Operation | Output shape |
|---|---|
| Input | 28 × 28 × 64 |
| Expansion 1×1 (t=4) | 28 × 28 × 256 |
| Depthwise 3×3, stride 2 | 14 × 14 × 256 |
| Projection 1×1 | 14 × 14 × 128 |

**Final output: 14 × 14 × 128**

### Stage 5 — Bottleneck Group 3

**Repetitions: 6 blocks**

| Operation | Output shape |
|---|---|
| Input | 14 × 14 × 128 |
| Expansion 1×1 (t=2) | 14 × 14 × 256 |
| Depthwise 3×3, stride 1 | 14 × 14 × 256 |
| Projection 1×1 | 14 × 14 × 128 |
| Residual addition | 14 × 14 × 128 |

All six blocks use stride 1.

**Final output: 14 × 14 × 128**

### Stage 6 — Bottleneck Group 4

**Repetitions: 1 block**

| Operation | Output shape |
|---|---|
| Input | 14 × 14 × 128 |
| Expansion 1×1 (t=4) | 14 × 14 × 512 |
| Depthwise 3×3, stride 2 | 7 × 7 × 512 |
| Projection 1×1 | 7 × 7 × 128 |

**Final output: 7 × 7 × 128**

### Stage 7 — Bottleneck Group 5

**Repetitions: 2 blocks**

| Operation | Output shape |
|---|---|
| Input | 7 × 7 × 128 |
| Expansion 1×1 (t=2) | 7 × 7 × 256 |
| Depthwise 3×3, stride 1 | 7 × 7 × 256 |
| Projection 1×1 | 7 × 7 × 128 |
| Residual addition | 7 × 7 × 128 |

**Final output: 7 × 7 × 128**

---

## 3. Final Feature Extraction Layers

### Stage 8 — 1×1 Convolution

**Input**

7 × 7 × 128

**Operation**

- Conv2D
- Kernel: 1×1
- Filters: 512

**Output**

7 × 7 × 512

### Stage 9 — Global Depthwise Convolution

**Input**

7 × 7 × 512

**Operation**

- GDConv
- Kernel: 7×7
- Depthwise filters: 512
- Stride: 1

**Output**

1 × 1 × 512

### Stage 10 — Final Linear Convolution

**Input**

1 × 1 × 512

**Operation**

- Conv2D
- Kernel: 1×1
- Filters: 128

**Output**

1 × 1 × 128

**Final facial embedding: 128 dimensions**

---

## 4. Final Architecture at a Glance

**112 × 112 × 3 → 128-D**

| Stage | Operation | Output |
|---|---|---|
| 0 | Input | 112 × 112 × 3 |
| 1 | Initial Conv | 56 × 56 × 64 |
| 2 | Initial DWConv | 56 × 56 × 64 |
| 3 | Bottleneck 1 (5×) | 28 × 28 × 64 |
| 4 | Bottleneck 2 (1×) | 14 × 14 × 128 |
| 5 | Bottleneck 3 (6×) | 14 × 14 × 128 |
| 6 | Bottleneck 4 (1×) | 7 × 7 × 128 |
| 7 | Bottleneck 5 (2×) | 7 × 7 × 128 |
| 8 | 1×1 Conv | 7 × 7 × 512 |
| 9 | GDConv 7×7 | 1 × 1 × 512 |
| 10 | Linear 1×1 Conv | 1 × 1 × 128 |

### Important Implementation Note

These dimensions describe one image in NHWC notation.

PyTorch normally uses NCHW, so the corresponding final tensor for a batch of 1 is:

`[1, 128, 1, 1]`

Usually flattened to:

`[1, 128]`

This is the facial embedding produced by the network.

---

**Implementation target:** This is the forward-pass architecture we'll implement in our project. 🔥

# Task 2.6 — Understanding ArcFace (Additive Angular Margin Loss)

## What is ArcFace?

ArcFace (Additive Angular Margin Loss) is a loss function used to train face recognition models.

Its main purpose is to make the model learn facial embeddings that:

- Have similar faces close together.
- Have different identities far apart.
- Produce highly discriminative facial features.

## 1. First, Understand Where ArcFace Fits

Your face recognition system has two major components:

**Input face image**

112 × 112 × 3

↓

**MobileFaceNet**

Feature extraction

↓

**Face embedding**

128 dimensions

↓

**ArcFace Loss**

Trains discriminative embeddings

MobileFaceNet extracts facial features. ArcFace helps train MobileFaceNet to distinguish identities effectively.

## 2. How Does ArcFace Work?

Imagine your dataset contains three people:

- Person A
- Person B
- Person C

MobileFaceNet generates an embedding for each face.

![ArcFace embedding distribution and angular margin illustration](https://images.openai.com/static-rsc-4/_gfXZmf0GlciEZcZD3eElq61ia5bpM9ZGWBtp72Lx3LuSLe2Bp09ghyoUYWpkBWW6Q0uvDcMRGMrG-A0_VeaJbluGnbAUxjHbyL3tdQkk274RdipbaLff1yB-ZQn4p8w1pyWcZUbsMja1fEjNcNQfD9KvyTVYso2cRWXzkoSBWM?purpose=inline)

[Image source: ArcFace — SR's Dev-Log](https://seungriyou.github.io/posts/arcface-additive-angular-margin-loss-for-deep-face-recognition/?utm_source=chatgpt.com)

ArcFace tries to organize these embeddings so that:

- Images belonging to Person A form a tight cluster.
- Images belonging to Person B form another cluster.
- Images belonging to Person C form another cluster.
- Different identity clusters are separated.

This is achieved using an angular margin.

## 3. What Makes ArcFace Different?

Traditional Softmax loss tries to classify identities correctly.

ArcFace adds an extra angular margin to make classification more challenging and encourage better separation.

Its central equation is:

$$
L=-\log
\frac{
e^{s\cos(\theta_y+m)}
}{
e^{s\cos(\theta_y+m)}
+
\sum_{j\ne y}e^{s\cos\theta_j}
}
$$

### Understanding the Symbols

| Symbol | Meaning |
|---|---|
| \(s\) | Scaling factor |
| \(m\) | Angular margin |
| \(\theta_y\) | Angle between embedding and correct identity's class weight |
| \(\theta_j\) | Angle for other identity classes |

**The key idea:** ArcFace adds margin \(m\) to the angle of the correct identity, making the model learn more discriminative facial embeddings.

## 4. How Will It Work in Your Project?

You have approximately 131,442 training images.

Suppose your dataset contains 5,000 unique identities.

During training:

1. MobileFaceNet receives a face image.
2. It generates a 128-dimensional embedding.
3. ArcFace compares this embedding against the identity class weights.
4. It calculates the angular-margin loss.
5. Backpropagation updates MobileFaceNet's weights.
6. The process repeats over training batches and epochs.

## 5. Important: Training vs Inference

| Training | Inference |
|---|---|
| MobileFaceNet | MobileFaceNet |
| ArcFace loss | Cosine similarity |
| Identity labels required | No identity labels required |
| Learns discriminative embeddings | Compares face embeddings |

At inference, you don't need the ArcFace classification head.

You simply:

1. Extract the embedding of Face A.
2. Extract the embedding of Face B.
3. Calculate cosine similarity.
4. Compare it against a chosen threshold.

### Remember This

- **MobileFaceNet** = Architecture (feature extractor).
- **ArcFace** = Loss function (training objective).
- **Cosine Similarity** = Metric (compares two embeddings).

This combination is a strong starting point for your face recognition project!

---

## 6. How ArcFace Organizes Same-Identity Embeddings

Suppose you have:

- 5,000 identities
- 25 images per identity
- 125,000 training images (approximately)

ArcFace trains MobileFaceNet to achieve two things.

### 6.1 Same Identity → Similar Embeddings

For example, Person A has 25 images:

- Image 1 → Embedding A1
- Image 2 → Embedding A2
- Image 3 → Embedding A3
- ...
- Image 25 → Embedding A25

ArcFace encourages all these embeddings to form a tight cluster in the feature space.

### 6.2 Different Identities → Separated Embeddings

Person B's embeddings should be separated from Person A's embeddings.

![Face recognition embeddings illustration](https://images.openai.com/static-rsc-4/HMTv4BbQO1k3ezj1hi4d-edCSxFg6uzMyedzWg9SRzy4EY5g2rXN8I3YSsKcnBMt3GqyihzOPKgvex_khGOSjWKQQzRx3bqFwaDvObGcLjSwOMxgkrdBUDWAiVjzQPoOUjC8hOUe1J4F4zQA4XNi4oPnHF3YYjWOFiJJmh43CfI?purpose=inline)

[Image source: Ars Futura — Face Recognition with FaceNet and MTCNN](https://arsfutura.com/blog/face-recognition-with-facenet-and-mtcnn?utm_source=chatgpt.com)

### The Ultimate Objective

| Comparison | Desired result |
|---|---|
| Person A image 1 vs Person A image 2 | High similarity |
| Person A image 1 vs Person A image 25 | High similarity |
| Person A image 1 vs Person B image 1 | Low similarity |

### Important Clarification

ArcFace doesn't directly compare all 25 images with each other.

It uses identity labels and class weights to train the embeddings through its angular-margin loss.

So the core understanding is:

**ArcFace's job is to help MobileFaceNet learn compact same-identity embeddings and well-separated different-identity embeddings.** ❤️

---

## 7. What You Should Remember

1. MobileFaceNet generates face embeddings.
2. ArcFace is a training loss function.
3. ArcFace uses an additive angular margin.
4. Identity labels are required during training.
5. ArcFace encourages better separation between identities.
6. The ArcFace classification head is not required during normal inference.
7. Cosine similarity can compare the resulting embeddings during face verification.